# BDIViz User Test Case 1

## Supplimentary materials

**GDC Metadata Validation Services:**
https://docs.gdc.cancer.gov/Data_Dictionary/gdcmvs/  
  
**Data Dictionary Viewer:**
https://docs.gdc.cancer.gov/Data_Dictionary/viewer/


---
  
#### Paper info: [link](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9929250/)

_**Multilevel proteomic analyses reveal molecular diversity between diffuse-type and intestinal-type gastric cancer**_  

_Shi W et. al_

Diffuse-type gastric cancer (DGC) and intestinal-type gastric cancer (IGC) are the major histological types of gastric cancer (GC). The molecular mechanism underlying DGC and IGC differences are poorly understood. In this research, we carry out multilevel proteomic analyses, including proteome, phospho-proteome, and transcription factor (TF) activity profiles, of 196 cases covering DGC and IGC in Chinese patients. Integrative proteogenomic analysis reveals ARIDIA mutation associated with opposite prognostic effects between DGC and IGC, via diverse influences on their corresponding proteomes. Systematical comparison and consensus clustering analysis identify three subtypes of DGC and IGC, respectively, based on distinct patterns of the cell cycle, extracellular matrix organization, and immune response-related proteins expression. TF activity-based subtypes demonstrate that the disease progressions of DGC and IGC were regulated by SWI/SNF and NFKB complexes. Furthermore, inferred immune cell infiltration and immune clustering show Th1/Th2 ratio is an indicator for immunotherapeutic effectiveness, which is validated in an independent GC anti-PD1 therapeutic patient group. Our multilevel proteomic analyses enable a more comprehensive understanding of GC and can further advance the precision medicine.


## 0. Load Dataset

In [1]:
import pandas as pd
import numpy as np

import panel as pn
import bdikit as bdi
from bdiviz import BDISchemaMatchingHeatMap

In [2]:
# Li GX et al.
Li = pd.read_csv("Li_GX_et_al_cleaned.csv").drop(columns=["Unnamed: 0"]).infer_objects()

# Shi W et al.
Shi = (
    pd.read_csv("Shi_W_et_al_cleaned.csv").drop(columns=["Unnamed: 0"]).infer_objects()
)

source = Li
target = "gdc"

In [3]:
source

,type,purity,ploidy,consent_age,stage,vital,grade,medical_history_bmi,medical_history_weight_at_time_of_surgery_kg,medical_history_alcohol_consumption,medical_history_tobacco_smoking_history,Tissue Type
0,Tumor,0.5370,2.000000,71,Stage I,Living,GX: Cannot be assessed,22.63,62.0,not reported,not reported,NaN
1,Tumor,0.6440,2.000000,39,unknown,Living,GX: Cannot be assessed,18.15,41.0,light drinker,past smoker unknown years,NaN
2,Tumor,0.3860,2.000000,54,Stage I,Living,GX: Cannot be assessed,18.48,54.0,not reported,not reported,NaN
3,Tumor,0.5700,2.010000,56,Stage III,Living,"G2: Nuclei slightly irregular, approximately 1...",59.88,169.0,light drinker,not reported,Solid Tissue Normal
4,Tumor,0.5700,2.010000,56,Stage III,Living,"G2: Nuclei slightly irregular, approximately 1...",59.88,169.0,light drinker,not reported,Primary Tumor
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Tumor,0.4875,1.901408,69,Stage I,Living,"G1: Nuclei round, uniform, approximately 10Âµm...",19.72,57.0,light drinker,past smoker more than 15 years,Solid Tissue Normal
96,Tumor,0.4875,1.901408,69,Stage I,Living,"G1: Nuclei round, uniform, approximately 10Âµm...",19.72,57.0,light drinker,past smoker more than 15 years,Primary Tumor
97,Tumor,0.4875,1.901408,69,Stage I,Living,"G1: Nuclei round, uniform, approximately 10Âµm...",19.72,57.0,light drinker,past smoker more than 15 years,Solid Tissue Normal
98,Tumor,0.4875,1.901408,69,Stage I,Living,"G1: Nuclei round, uniform, approximately 10Âµm...",19.72,57.0,light drinker,past smoker more than 15 years,Primary Tumor


## 1. Visualization Interface

In [4]:
heatmap_manager = BDISchemaMatchingHeatMap(
    source,
    target=target,
    top_k=20,
    # additional_sources=additional_sources,
    ai_assitant=False,
)
heatmap_manager.plot_heatmap()

Extracting features from 12 columns...


  0%|          | 0/12 [00:00<?, ?it/s]

Table features loaded for 734 columns


Column(scroll=True)
    [0] Row(styles={'background': '...}, width=1200)
        [0] Select(name='Source Column', options=['type', 'purity', ...], value='type', width=120)
        [1] Select(name='Candidate Type', options=['All', 'enum', ...], value='All', width=120)
        [2] IntSlider(end=5, name='Similar Sources', width=150)
        [3] FloatSlider(name='Candidate Threshold', step=0.01, value=0.1, width=150)
        [4] Column
            [0] Button(button_type='success', name='Accept Match')
            [1] Button(button_type='danger', name='Reject Match')
            [2] Button(button_type='warning', name='Discard Column')
        [5] Column
            [0] Button(button_style='outline', button_type='warning', name='Undo')
            [1] Button(button_style='outline', button_type='primary', name='Redo')
        [6] Column
            [0] Button(button_type='primary', name='Show/Hide AI Assistant')
            [1] Button(button_type='primary', name='Show/Hide Operation Log')
    [1] ParamFunction(function, _pane=Str, defer_load=False)
    [2] Spacer(height=5)
    [3] Column
        [0] ParamFunction(function, _pane=Column, defer_load=False)

## 2. Schema Matching with BDIViz Results

In [6]:
from bdikit.mapping_algorithms.column_mapping.algorithms import TwoPhaseSchemaMatcher

two_phase_viz = TwoPhaseSchemaMatcher(top_k_matcher=heatmap_manager)
column_mappings = bdi.match_schema(source, target=target, method=two_phase_viz)
column_mappings

,source,target
0,type,tissue_type
1,purity,tumor_purity
2,ploidy,ploidy
3,consent_age,undescended_testis_corrected_age
4,stage,irs_stage
5,vital,vital_status
6,grade,demographics
7,medical_history_bmi,risk_factors
8,medical_history_weight_at_time_of_surgery_kg,alcohol_drinks_per_day
9,medical_history_alcohol_consumption,


## 3. Value Matchings

In [9]:
column_mappings = column_mappings[column_mappings["target"].str.strip().astype(bool)]
mappings = bdi.match_values(
    source,
    column_mapping=column_mappings,
    target=target,
    method="tfidf",
)

for mapping in mappings:
    print(f"{mapping.attrs['source']} => {mapping.attrs['target']}")
    display(mapping)
    print("")

type => tissue_type


,source,target,similarity
0,Tumor,Tumor,1.0



stage => irs_stage


,source,target,similarity
0,unknown,Unknown,1.0
1,Stage IV,NaN,NaN
2,Stage I,NaN,NaN
3,Stage III,NaN,NaN
4,Stage II,NaN,NaN



vital => vital_status


...

,source,target,similarity
0,"G4: Nuclei bizarre and multilobulated, 20Âµm o...",Undifferentiated or Poorly Differentiated,0.371
1,"G3: Nuclei very irregular, approximately 20Âµm...",Undifferentiated or Poorly Differentiated,0.328
2,"G2: Nuclei slightly irregular, approximately 1...",Poorly Differentiated,0.302
3,GX: Cannot be assessed,NaN,NaN
4,"G1: Nuclei round, uniform, approximately 10Âµm...",NaN,NaN



medical_history_alcohol_consumption => alcohol_history


,source,target,similarity
0,not reported,Not Reported,1.000
1,non-drinker,no,0.317
2,light drinker,NaN,NaN



medical_history_tobacco_smoking_history => tobacco_smoking_status


,source,target,similarity
0,not reported,Not Reported,1.000
1,current smoker,Current Smoker,1.000
2,past smoker unknown years,Unknown,0.642
3,non-smoker,Lifelong Non-Smoker,0.631
4,past smoker more than 15 years,Current Reformed Smoker for < or = 15 yrs,0.429
5,past smoker within 15 years,Smoker at Diagnosis,0.346
